# GSE221156 (human pancreatic islets): Seurat object for donors **without diabetes (ND)** and **with type 2 diabetes (T2D)**, with BMI

**Author's note (junior bioinformatician).** This notebook builds one Seurat object in which **every cell carries its library, its
donor, its group (ND or T2D) and the donor's BMI**, and it sorts the donors into **lean** and **not lean**. The comparison is
**no diabetes (ND, 17 donors) versus type 2 diabetes (T2D, 17 donors)**. The 14 pre-diabetic (PD) donors are left out by default
(`groups_to_keep`), because PD is a third, intermediate state and not a clean control.

## Why ND versus T2D is the better contrast
* The paper picked the ND donors to be **similar to the T2D donors in age, sex, BMI and ancestry** (Bandesh et al., *EMBO J* 2026), and HbA1c is
  clearly different (ND mean 5.2%, T2D mean 7.6%). That is what a control group should look like.
* **All 17 ND donors have a library of their own**, so no barcode decoding is needed for the control group.
* The paper found that PD looked close to ND in beta cells, so a PD-versus-T2D contrast would blur the difference you want.

## The one problem that remains: three T2D donors sit inside pooled libraries
Of the 54 GEO libraries, **12 are pools of two donors** (titles like `Islet70/71_MS19034`). Your earlier notebook matched titles exactly and
therefore found **14 T2D** libraries instead of 17 donors. The three missing T2D donors, **Islet71, Islet84 and Islet119**, exist **only** inside pools
(each shared with a PD donor: 70/71, 84/85, 118/119). The authors separated them with **Demuxlet**, which reads donor SNP genotypes. Those genotypes
and the per-barcode assignments are **not in the GEO files** (the RAW tar holds count matrices only), and **I cannot work out the donor of a
barcode from the counts alone**.

So the honest numbers are:

| | ND | T2D |
|---|---|---|
| Donors expected | 17 | 17 |
| Donors with their own library (usable now) | **17** | **14** |
| Donors only in pools (need the barcode table) | 0 | 3 (Islet71, 84, 119) |

If you cannot get the barcode table (section 5 says where to look), run the comparison as **17 ND versus 14 T2D donors** and say so.
The other pools (47/48, 57/58, 59/60) mix an ND or PD donor with a T2D, ND or PD donor who also has a library of their own, so they add nothing the
notebook needs, and the notebook does not read them.

## Other things to know
* **This is pancreatic islets, not adipose tissue.** The endothelial cells here (4,383 in the whole study, paper Figure 1C) are islet capillary cells. An ERG result
  from this dataset says nothing about fat.
* **BMI is read from the GEO sample records** (the `bmi:` field; your own output shows 35.6 for MS17001). Table EV1 of the paper can fill gaps.
* **With ND as the control, a lean group is more realistic.** The paper's Figure 1B shows ND BMI spanning roughly 20 to 40, so a 2 x 2 design
  (ND or T2D, lean or not lean) may have a few donors in each box. Check the table in section 6 before planning any lean-specific test.
* **HbA1c is a cross-check on the labels.** The notebook flags any donor whose HbA1c does not fit its group (for example an ND donor at 5.7% or above).

## Stages
1 settings · 2 unpack · 3 read GEO sample records · 4 library and donor audit · 5 decode the barcodes of pooled libraries ·
6 BMI, lean / not lean, label checks · 7 build the Seurat object · 8 checks and save.

*Caution:* I could not run this on the real files from my workspace (the tar is 2.9 GB). The logic was tested on a small made-up copy of the file
layout only. Read every table before you trust it.

## 1. Settings

**Cell annotation:** Paths, the groups to keep, and the BMI cut-offs. The two cut-offs are the standard adult WHO ones (lean below 25, obese 30 and above); change them if your supervisor uses others.

In [ ]:
tar_file   <- "C:/Users/user/Downloads/GSE221156_RAW.tar"
raw_dir    <- "C:/Users/user/Downloads/GSE221156_RAW"     # where the .mtx.gz / .tsv.gz files are (or will be unpacked)
out_dir    <- "GSE221156_results"

groups_to_keep <- c("ND", "T2D")     # ND = no diabetes (control), T2D = type 2 diabetes. Add "PD" to keep the 14 pre-diabetic donors too.

# BMI classes (kg/m2): lean < lean_cutoff <= overweight < obese_cutoff <= obese. "Not lean" = overweight + obese.
lean_cutoff  <- 25
obese_cutoff <- 30

# Section 5: a table that says which donor each barcode of a POOLED library came from (see the notes there). Leave "" if you do not have it.
demux_file         <- ""
demux_library_col  <- "library_id"   # e.g. MS19034
demux_barcode_col  <- "barcode"      # e.g. AAACCTGAGGCAGGTT or AAACCTGAGGCAGGTT-1
demux_donor_col    <- "donor"        # e.g. Islet70 or 70

# Optional: the paper's Table EV1 (donor characteristics) saved as .csv, used to fill missing BMI
ev1_file       <- ""
ev1_donor_col  <- "donor"
ev1_bmi_col    <- "bmi"

geo_meta_csv <- file.path(out_dir, "geo_sample_metadata.csv")   # the GEO records are saved here after the first download
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

**Cell annotation:** Load the packages. GEOquery is only needed once, to download the sample records.

In [ ]:
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
if (!file.exists(geo_meta_csv) && !requireNamespace("GEOquery", quietly = TRUE)) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
  BiocManager::install("GEOquery")
}
cat("Seurat", as.character(packageVersion("Seurat")), "\n")

## 2. Unpack the archive (only if needed)

**Cell annotation:** Unpack the .tar into `raw_dir` if the files are not there yet. Each library has three files: `_matrix.mtx.gz`, `_features.tsv.gz`, `_barcodes.tsv.gz` (3 x 54 = 162 files).

In [ ]:
if (length(list.files(raw_dir, pattern = "_matrix\\.mtx\\.gz$")) == 0) {
  dir.create(raw_dir, showWarnings = FALSE, recursive = TRUE)
  untar(tar_file, exdir = raw_dir)
}
mtx_files <- list.files(raw_dir, pattern = "_matrix\\.mtx\\.gz$", full.names = TRUE, recursive = TRUE)
cat("libraries found:", length(mtx_files), "(expected 54)\n")
stopifnot(length(mtx_files) > 0)

*What you should see:* 54 libraries.

## 3. The GEO sample records (title, donor characteristics, BMI)

**Cell annotation:** Download the 54 sample records once and keep, for each: its GSM accession, its title, and the characteristics text (which holds disease state, sex, BMI, HbA1c, age, race and library id). They are saved to a .csv so the next run is offline.

In [ ]:
get_geo_meta <- function() {
  if (file.exists(geo_meta_csv)) return(read.csv(geo_meta_csv, stringsAsFactors = FALSE))
  gse <- GEOquery::getGEO("GSE221156", GSEMatrix = FALSE)
  gsm <- GEOquery::GSMList(gse)
  m <- do.call(rbind, lapply(names(gsm), function(id) {
    mt <- GEOquery::Meta(gsm[[id]])
    data.frame(gsm = id, title = as.character(mt$title)[1],
               characteristics = paste(mt$characteristics_ch1, collapse = " || "), stringsAsFactors = FALSE)
  }))
  write.csv(m, geo_meta_csv, row.names = FALSE); m
}
geo <- get_geo_meta()
cat("GEO sample records:", nrow(geo), "\n")
# turn "key: value || key: value" into one column per key
parse_chars <- function(x) {
  kv <- strsplit(x, " \\|\\| ")[[1]]
  k <- tolower(gsub("[^A-Za-z0-9]+", "_", sub(":.*$", "", kv))); v <- trimws(sub("^[^:]*:", "", kv))
  stats::setNames(as.list(v), k)
}
ch <- lapply(geo$characteristics, parse_chars)
keys <- unique(unlist(lapply(ch, names)))
for (k in keys) geo[[k]] <- vapply(ch, function(z) if (is.null(z[[k]])) NA_character_ else z[[k]], "")
names(geo)[names(geo) == "library_id"] <- "geo_library_id"   # keep it apart from our own library_id
keys <- sub("^library_id$", "geo_library_id", keys)
cat("characteristic fields found:", paste(keys, collapse = ", "), "\n")
print(head(geo[, intersect(c("gsm", "title", "disease_state", "sex", "bmi", "hba1c", "age"), colnames(geo))], 8))
cat("\nThe POOLED libraries (title has a slash). Look at what their characteristics say:\n")
print(geo[grepl("/", geo$title), intersect(c("gsm", "title", "disease_state", "sex", "bmi", "age", "library_id"), colnames(geo))])

*What you should see:* The field names (including `bmi`), the first rows, and the pooled libraries. If the pooled rows show one BMI, it may describe only one of the two donors: do not use it for both.

## 4. Library and donor audit
The study lists its donors by group on the GEO series page. I copied those lists here. A library is **single-donor** if its
title is `IsletNN_MSxxxxx`, and **pooled** if it is `IsletNN/MM_MSxxxxx`.

**Cell annotation:** Donor lists (from the GEO series page: 17 ND, 14 PD, 17 T2D), the library table, and the check of which donors appear only inside pools.

In [ ]:
nd_ids  <- c(29, 34, 37, 38, 42, 45, 47, 54, 57, 59, 61, 68, 73, 80, 104, 107, 123)
pd_ids  <- c(40, 50, 53, 55, 56, 58, 62, 63, 70, 85, 89, 101, 118, 127)
t2d_ids <- c(28, 30, 31, 32, 33, 39, 41, 44, 48, 52, 60, 71, 84, 91, 126, 119, 128)
stopifnot(length(nd_ids) == 17, length(pd_ids) == 14, length(t2d_ids) == 17)
donor_group <- data.frame(donor = paste0("Islet", c(nd_ids, pd_ids, t2d_ids)),
                          group = rep(c("ND", "PD", "T2D"), c(17, 14, 17)), stringsAsFactors = FALSE)
donor_group$group_label <- c(ND = "No diabetes", PD = "Pre-diabetes", T2D = "Type 2 diabetes")[donor_group$group]

lib <- data.frame(file_prefix = sub("_matrix\\.mtx\\.gz$", "", mtx_files), stringsAsFactors = FALSE)
lib$sample_id  <- basename(lib$file_prefix)
lib$gsm        <- sub("_.*$", "", lib$sample_id)
lib$library_id <- sub("^GSM[0-9]+_", "", lib$sample_id)
lib <- merge(lib, geo, by = "gsm", all.x = TRUE, sort = FALSE)
stopifnot(!anyNA(lib$title))

nums <- regmatches(lib$title, gregexpr("(?<=Islet)[0-9]+|(?<=/)[0-9]+", lib$title, perl = TRUE))
lib$donors_in_library <- vapply(nums, function(z) paste0("Islet", z, collapse = "+"), "")
lib$pooled <- vapply(nums, length, 1L) > 1
lib$library_donor <- ifelse(lib$pooled, NA_character_, lib$donors_in_library)
lib$library_group <- donor_group$group[match(lib$library_donor, donor_group$donor)]
cat("single-donor libraries:", sum(!lib$pooled), "| pooled libraries:", sum(lib$pooled), "(expected 12)\n\n")

solo_donors <- unique(na.omit(lib$library_donor))
cov <- donor_group; cov$has_own_library <- cov$donor %in% solo_donors
cov$pooled_libraries <- vapply(cov$donor, function(d) paste(lib$library_id[lib$pooled & grepl(paste0("(^|\\+)", d, "(\\+|$)"), lib$donors_in_library)], collapse = ","), "")
cat("Donors that have a library of their own, by group:\n"); print(table(cov$group, cov$has_own_library))
cat("\nDonors of the groups you keep that appear ONLY in pooled libraries (they need the barcode table in section 5):\n")
print(cov[!cov$has_own_library & cov$group %in% groups_to_keep, ], row.names = FALSE)
cat("\nWhat the pooled libraries mix:\n")
print(lib[lib$pooled, c("library_id", "title", "donors_in_library")], row.names = FALSE)

*What you should see:* About 42 single-donor and 12 pooled libraries. All 17 ND donors have their own library. Among T2D, 14 donors do and 3 (Islet71, 84, 119) do not. (Among PD, 11 do and 3, Islet70, 85 and 118, do not; PD is not kept by default.) The 14 T2D libraries are the 14 your earlier notebook found.

## 5. Decode the barcodes of the pooled libraries
For a single-donor library every cell belongs to that donor, so no decoding is needed. **For ND versus T2D only three pooled libraries matter**
(`Islet70/71`, `Islet84/85`, `Islet118/119`, each holding one T2D donor and one PD donor); the donor of each barcode in them must come from a table. **I could not see such a table on the GEO page** (the only file there is the RAW tar of
count matrices). Possible sources, in the order I would try them:

1. **The authors' processed object or Zenodo record.** The paper's data statement points to Zenodo record 14656366 (`Rmarkdown.zip`, 1.2 GB,
   contents not listed on its page) and to a CELLxGENE collection. Processed objects like these usually hold a donor per cell. I have not opened them.
2. **The corresponding author** (Efthymios Motakis, listed as the GEO contact), who can share the Demuxlet `.best` files.
3. **Doing it yourself** needs the donors' genotypes and the sequencing reads (BAM files from SRA, PRJNA913127). Demuxlet reads SNPs from the reads.
   The genotypes are from de-identified donors and are not public. So this is not a route open to you on the count matrices alone.

**What the notebook does with what you have.** If `demux_file` is set, it needs one row per barcode with the library id, the barcode and the donor
(`Islet70`, or just `70`). A value that starts with `DBL` is treated as a doublet and dropped, and a Demuxlet `SNG-...` value is read by its digits.
If `demux_file` is empty, the cells of pooled libraries are **left out and counted**, never guessed. Cells the table assigns to a PD donor are dropped later, because PD is not in `groups_to_keep`.

**Cell annotation:** Read the barcode table if you have one. Barcodes are compared without the `-1` suffix and without any library prefix, so `GSM..._MS19034_AAACC...-1`, `MS19034_AAACC...` and `AAACC...` all match.

In [ ]:
norm_bc  <- function(x) sub("-[0-9]+$", "", sub("^.*_", "", as.character(x)))
norm_don <- function(x) { x <- as.character(x); ifelse(grepl("^DBL", x), "Doublet", ifelse(grepl("[0-9]", x), paste0("Islet", sub("^[^0-9]*([0-9]+).*$", "\\1", x)), NA_character_)) }
demux <- NULL
if (nzchar(demux_file)) {
  demux <- read.delim(demux_file, stringsAsFactors = FALSE, check.names = FALSE, sep = if (grepl("\\.csv(\\.gz)?$", demux_file)) "," else "\t")
  need <- c(demux_library_col, demux_barcode_col, demux_donor_col)
  stopifnot(all(need %in% colnames(demux)))
  demux <- data.frame(library_id = sub("^GSM[0-9]+_", "", as.character(demux[[demux_library_col]])),
                      bc = norm_bc(demux[[demux_barcode_col]]), donor = norm_don(demux[[demux_donor_col]]), stringsAsFactors = FALSE)
  demux$key <- paste(demux$library_id, demux$bc, sep = "|")
  cat("barcode table:", nrow(demux), "rows\n"); print(table(demux$library_id[demux$donor == "Doublet"]))
  # every assigned donor must be one of the two donors of that library
  ok <- mapply(function(l, d) is.na(d) || d == "Doublet" || grepl(d, lib$donors_in_library[match(l, lib$library_id)], fixed = TRUE), demux$library_id, demux$donor)
  cat("assignments inconsistent with the library's two donors (set to 'Inconsistent' and dropped):", sum(!ok), "\n")
  demux$donor[!ok] <- "Inconsistent"
} else cat("No barcode table: pooled libraries will be left out.\n")

*What you should see:* Either the table summary, or the message that pooled libraries will be left out. A non-zero count of inconsistent assignments means the table does not match these libraries.

## 6. BMI and the lean / not lean classes
BMI comes from the GEO `bmi` field of each donor's **own** library. A donor with no own library has no BMI from GEO and is filled from the paper's
Table EV1 (`ev1_file`) if you give it; otherwise it stays `NA` and is **not** classed.

**Cell annotation:** Build the donor table: group, sex, age, HbA1c, BMI, the BMI source, and the class. Lean is BMI below the cut-off; "not lean" is everything from the cut-off up.

In [ ]:
solo <- lib[!lib$pooled, ]
num <- function(x) suppressWarnings(as.numeric(gsub("[^0-9.]", "", x)))
dn <- data.frame(donor = solo$library_donor, group = solo$library_group,
                 sex = if ("sex" %in% names(solo)) solo$sex else NA, age = num(solo$age), hba1c = num(solo$hba1c),
                 bmi = num(solo$bmi), bmi_source = ifelse(is.na(num(solo$bmi)), NA, "GEO sample record"),
                 race = if ("race" %in% names(solo)) solo$race else NA, stringsAsFactors = FALSE)
dn <- dn[!duplicated(dn$donor), ]
dn <- merge(donor_group, dn[, setdiff(names(dn), "group")], by = "donor", all.x = TRUE, sort = FALSE)

if (nzchar(ev1_file)) {
  ev <- read.csv(ev1_file, stringsAsFactors = FALSE, check.names = FALSE)
  ev$donor <- norm_don(ev[[ev1_donor_col]]); ev$bmi_ev1 <- num(ev[[ev1_bmi_col]])
  dn$bmi_ev1 <- ev$bmi_ev1[match(dn$donor, ev$donor)]
  clash <- which(!is.na(dn$bmi) & !is.na(dn$bmi_ev1) & abs(dn$bmi - dn$bmi_ev1) > 0.5)
  if (length(clash)) { cat("BMI differs between GEO and Table EV1 for:\n"); print(dn[clash, c("donor", "bmi", "bmi_ev1")]) }
  fill <- is.na(dn$bmi) & !is.na(dn$bmi_ev1); dn$bmi[fill] <- dn$bmi_ev1[fill]; dn$bmi_source[fill] <- "paper Table EV1"
}
dn$bmi_class <- cut(dn$bmi, c(-Inf, lean_cutoff, obese_cutoff, Inf), right = FALSE, labels = c("Lean", "Overweight", "Obese"))
dn$lean_status <- ifelse(is.na(dn$bmi), NA, ifelse(dn$bmi < lean_cutoff, "Lean", "Not lean"))
dn <- dn[order(dn$group, as.integer(sub("Islet", "", dn$donor))), ]
write.csv(dn, file.path(out_dir, "donor_table_BMI.csv"), row.names = FALSE)
print(dn[dn$group %in% groups_to_keep, ], row.names = FALSE)
cat("\nDonors per group and BMI class (NA = no BMI available):\n"); print(table(dn$group, dn$lean_status, useNA = "ifany"))
cat("\nDonors without a BMI:", paste(dn$donor[is.na(dn$bmi) & dn$group %in% groups_to_keep], collapse = ", "), "\n")

*What you should see:* One row per donor with BMI and class, then a donor count by group and lean status. The six pool-only donors show NA unless Table EV1 was supplied. Few donors are expected to be lean: the paper reports BMI mostly between 25 and 40 in PD and T2D.

**Cell annotation:** Check the labels and the matching between the groups. (1) HbA1c against group; (2) GEO's own `disease_state` against the donor list; (3) BMI, age and sex of ND against T2D, because ND was meant to be matched to T2D.

In [ ]:
k <- dn[dn$group %in% groups_to_keep, ]
cat("HbA1c by group (the paper reports ND mean 5.2%, T2D mean 7.6%):\n"); print(tapply(k$hba1c, k$group, function(x) round(c(n = sum(!is.na(x)), mean = mean(x, na.rm = TRUE), min = min(x, na.rm = TRUE), max = max(x, na.rm = TRUE)), 2)))
odd <- k[(k$group == "ND" & !is.na(k$hba1c) & k$hba1c >= 5.7) | (k$group == "T2D" & !is.na(k$hba1c) & k$hba1c < 5.7), c("donor", "group", "hba1c")]
cat("\nDonors whose HbA1c does not fit their group (check them against the paper):\n"); print(odd, row.names = FALSE)
if ("disease_state" %in% names(solo)) {
  gs <- ifelse(grepl("type ?2|t2d", solo$disease_state, ignore.case = TRUE), "T2D", ifelse(grepl("pre", solo$disease_state, ignore.case = TRUE), "PD", "ND"))
  bad <- solo[gs != solo$library_group, c("library_id", "title", "disease_state", "library_group")]
  cat("\nLibraries where GEO's disease_state disagrees with the donor list:", nrow(bad), "\n"); if (nrow(bad)) print(bad, row.names = FALSE)
}
cat("\nBMI, age and sex, ND vs T2D (medians; Wilcoxon p for BMI and age):\n")
print(aggregate(cbind(bmi, age) ~ group, data = k, FUN = function(x) round(median(x), 1)))
if (all(c("ND", "T2D") %in% k$group)) {
  for (v in c("bmi", "age")) { a <- k[[v]][k$group == "ND"]; b <- k[[v]][k$group == "T2D"]
    if (sum(!is.na(a)) > 2 && sum(!is.na(b)) > 2) cat(v, ": p =", signif(wilcox.test(a, b)$p.value, 3), "\n") }
  print(table(k$group, k$sex, useNA = "ifany"))
}

*What you should see:* HbA1c summaries, a (hopefully empty) list of conflicting donors, and a comparison of BMI, age and sex between ND and T2D. A small p for BMI means the groups are not BMI-matched, so BMI should be a covariate in later tests.

## 7. Build the Seurat object

**Cell annotation:** Read the libraries to keep: the single-donor libraries of the chosen groups, plus the pooled libraries if a barcode table was given. Cell names are `library_id_barcode`, so they cannot collide across libraries.

In [ ]:
# single-donor libraries of the kept groups; plus the pooled libraries that hold a kept donor who has no library of his or her own (needs the barcode table)
need_donors <- cov$donor[cov$group %in% groups_to_keep & !cov$has_own_library]
pool_needed <- lib$pooled & vapply(lib$donors_in_library, function(z) any(strsplit(z, "\\+")[[1]] %in% need_donors), TRUE)
want <- lib[(!lib$pooled & lib$library_group %in% groups_to_keep) | (pool_needed & !is.null(demux)), ]
cat("libraries to read:", nrow(want), "(", sum(!want$pooled), "single-donor +", sum(want$pooled), "pooled )\n")
cat("pooled libraries that hold a kept donor with no library of his or her own:", paste(lib$library_id[pool_needed], collapse = ", "), "\n")
if (any(pool_needed) && is.null(demux)) cat("-> no barcode table: those libraries are NOT read, so the donors", paste(need_donors, collapse = ", "), "are missing.\n")
read_lib <- function(prefix, id) {
  m <- as(readMM(paste0(prefix, "_matrix.mtx.gz")), "CsparseMatrix")
  f <- read.delim(paste0(prefix, "_features.tsv.gz"), header = FALSE, stringsAsFactors = FALSE)
  b <- readLines(gzfile(paste0(prefix, "_barcodes.tsv.gz")))
  stopifnot(ncol(m) == length(b), nrow(m) == nrow(f))
  colnames(m) <- paste(id, b, sep = "_"); list(m = m, f = f)
}
mats <- vector("list", nrow(want)); feat <- NULL
for (i in seq_len(nrow(want))) {
  r <- read_lib(want$file_prefix[i], want$library_id[i])
  if (is.null(feat)) feat <- r$f else stopifnot(identical(feat[[1]], r$f[[1]]))   # same genes, same order, in every library
  mats[[i]] <- r$m
}
counts <- do.call(cbind, mats); rm(mats); invisible(gc())
rownames(counts) <- make.unique(feat[[2]])
cat("matrix:", nrow(counts), "genes x", ncol(counts), "cells\n")

*What you should see:* A genes x cells matrix. For the 31 single-donor ND and T2D libraries, expect very roughly 200,000 to 250,000 cells (the paper reports 88,521 ND and 77,782 T2D cells after its own QC, and these raw counts are higher).

**Cell annotation:** Attach the metadata to each cell: library, pooled or not, donor (from the library for single-donor libraries; from the barcode table for pooled ones), group, and the donor's BMI and class.

In [ ]:
cell <- data.frame(cell = colnames(counts), stringsAsFactors = FALSE)
cell$library_id <- sub("_[^_]+$", "", cell$cell)
cell$barcode    <- sub("^.*_", "", cell$cell)
li <- match(cell$library_id, lib$library_id)
cell$gsm        <- lib$gsm[li]
cell$library_pooled <- lib$pooled[li]
cell$donor      <- lib$library_donor[li]                      # single-donor libraries
if (!is.null(demux)) {
  pi <- which(cell$library_pooled)
  hit <- match(paste(cell$library_id[pi], norm_bc(cell$barcode[pi]), sep = "|"), demux$key)
  cell$donor[pi] <- ifelse(is.na(hit), "Unassigned", demux$donor[hit])
}
cell$donor_status <- ifelse(is.na(cell$donor), "Unassigned", ifelse(cell$donor %in% c("Unassigned", "Doublet", "Inconsistent"), cell$donor, "Assigned"))
cat("Cells by donor status:\n"); print(table(cell$donor_status, cell$library_pooled))

di <- match(cell$donor, dn$donor)
cell$group       <- dn$group[di]
cell$group_label <- dn$group_label[di]
cell$sex         <- dn$sex[di];   cell$age <- dn$age[di];  cell$hba1c <- dn$hba1c[di]
cell$bmi         <- dn$bmi[di];   cell$bmi_class <- as.character(dn$bmi_class[di]); cell$lean_status <- dn$lean_status[di]
cell$bmi_source  <- dn$bmi_source[di]
rownames(cell) <- cell$cell
keep <- cell$donor_status == "Assigned" & cell$group %in% groups_to_keep
cat("cells kept:", sum(keep), "of", nrow(cell), "\n")
obj <- CreateSeuratObject(counts = counts[, keep], meta.data = cell[keep, setdiff(names(cell), "cell")], project = "GSE221156", min.cells = 0, min.features = 0)
rm(counts); invisible(gc())
obj <- PercentageFeatureSet(obj, pattern = "^MT-", col.name = "percent.mt")
obj$group_lean <- paste(obj$group, obj$lean_status, sep = "_")      # e.g. ND_Lean, T2D_Not lean
obj

*What you should see:* A Seurat object, and a table of how many cells were assigned, unassigned or doublets.

## 8. Checks and save

**Cell annotation:** Does the object contain the groups you expected? **Target: 17 ND and 17 T2D donors.** Anything less is listed with the reason.

In [ ]:
chk <- unique(obj@meta.data[, c("donor", "group", "bmi", "lean_status", "bmi_source")])
tab <- table(chk$group); print(tab)
for (g in groups_to_keep) {
  target <- c(ND = 17, PD = 14, T2D = 17)[g]
  if (tab[g] < target) {
    miss <- setdiff(dn$donor[dn$group == g], chk$donor)
    cat("\n", g, ": only", tab[g], "of", target, "donors in the object. Missing:", paste(miss, collapse = ", "),
        "\n   (a donor with no library of his or her own lives only in a pool and needs the barcode table in section 5)\n")
  }
}
cat("\nDonors by group and lean status in the object:\n"); print(table(chk$group, chk$lean_status, useNA = "ifany"))
cat("\nCells per donor (smallest five):\n"); print(head(sort(table(obj$donor)), 5))
cat("\nCells from pooled libraries in the object (only the assigned T2D donors should appear):", sum(obj$library_pooled), "\n")
print(table(obj$group, obj$library_pooled))

*What you should see:* Counts of donors per group. Without the barcode table: 17 ND and 14 T2D. With it: 17 and 17.

**Cell annotation:** Save the object and the tables.

In [ ]:
saveRDS(obj, file.path(out_dir, "GSE221156_ND_T2D_seurat.rds"))
write.csv(obj@meta.data, file.path(out_dir, "cell_metadata.csv"))
write.csv(lib[, c("gsm", "library_id", "title", "pooled", "donors_in_library", "library_group")], file.path(out_dir, "library_table.csv"), row.names = FALSE)
writeLines(capture.output(sessionInfo()), file.path(out_dir, "sessionInfo.txt"))
list.files(out_dir)

## What comes next
This notebook stops at the annotated object. The next steps follow the earlier notebooks: QC per library (the paper used SoupX for ambient RNA, Scrublet and DoubletFinder for
doublets, and removed cells with more than 40% mitochondrial reads in beta cells and 20% in other cell types), clustering, finding the endothelial cluster (`PLVAP` is
the paper's endothelial marker, with `PECAM1`/`CDH5` kept out of the naming), and pseudobulk per **donor** with edgeR: **T2D versus ND**, adjusting for sex, age and BMI when the numbers allow.
For the lean question, test T2D versus ND within the lean and the not-lean donors only if both boxes hold enough donors (see section 6).

**Limits.** Donor numbers, not cell numbers, set the power. With 17 donors per group, and about 90 endothelial cells per donor on average (the paper's 4,383 islet endothelial cells over 48 donors), ERG may be barely detected in many donors.
Keep the per-donor read counts next to every ERG result. Do not call a result about this dataset a result about adipose tissue.